In [ ]:
import math
import os
import random
import shutil
import time
import gc
from contextlib import nullcontext
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, models, transforms

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
)
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())


PyTorch: 2.11.0+cu128
CUDA available: True


In [ ]:
SEED = 42


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True


set_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device


device(type='cuda')

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Google Drive mount skipped:', exc)


DATA_ROOT = Path('/content/drive/MyDrive/Semester 2/pemsin/Project/dataset')

OUTPUT_DIR = Path('/content/outputs_all_models_focal')
CHECKPOINT_DIR = OUTPUT_DIR / 'checkpoints'
PLOT_DIR = OUTPUT_DIR / 'plots'
REPORT_DIR = OUTPUT_DIR / 'reports'

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)


def resolve_dataset_folder(folder_name):
    candidates = [
        DATA_ROOT / folder_name,
        Path('/content/dataset') / folder_name,
        Path('/content') / folder_name,
        Path('dataset') / folder_name,
        Path(folder_name),
        Path(r'G:\My Drive\Semester 2\pemsin\Project\dataset') / folder_name,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    searched = '\\n'.join(str(candidate) for candidate in candidates)
    raise FileNotFoundError(f'Folder "{folder_name}" not found. Checked:\\n{searched}')


COPY_DATA_TO_LOCAL = True
LOCAL_DATA_ROOT = Path('/content/working_dataset')


def copy_to_local(source_dir):
    source_dir = Path(source_dir)
    source_text = str(source_dir)
    if not COPY_DATA_TO_LOCAL:
        return source_dir
    if not source_text.startswith('/content/drive/'):
        return source_dir

    destination_dir = LOCAL_DATA_ROOT / source_dir.name
    if destination_dir.exists():
        print(f'Using existing local copy: {destination_dir}')
        return destination_dir

    print(f'Copying {source_dir} to {destination_dir}. This is slow once, but training epochs are much faster after it finishes.')
    destination_dir.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(source_dir, destination_dir)
    return destination_dir


TRAIN_DIR = copy_to_local(resolve_dataset_folder('Augmented Images'))
EVAL_DIR = copy_to_local(resolve_dataset_folder('Original Images'))

print('Training folder:       ', TRAIN_DIR)
print('Validation/test folder:', EVAL_DIR)
print('Output folder:         ', OUTPUT_DIR)


Mounted at /content/drive
Copying /content/drive/MyDrive/Semester 2/pemsin/Project/dataset/Augmented Images to /content/working_dataset/Augmented Images. This is slow once, but training epochs are much faster after it finishes.
Copying /content/drive/MyDrive/Semester 2/pemsin/Project/dataset/Original Images to /content/working_dataset/Original Images. This is slow once, but training epochs are much faster after it finishes.
Training folder:        /content/working_dataset/Augmented Images
Validation/test folder: /content/working_dataset/Original Images
Output folder:          /content/outputs_all_models_focal


In [ ]:
train_raw_dataset = datasets.ImageFolder(root=TRAIN_DIR)
eval_raw_dataset = datasets.ImageFolder(root=EVAL_DIR)

class_names = train_raw_dataset.classes
num_classes = len(class_names)

if eval_raw_dataset.classes != class_names:
    raise ValueError(
        'Class folders in Augmented Images and Original Images do not match.\n'
        f'Augmented classes: {train_raw_dataset.classes}\n'
        f'Original classes:  {eval_raw_dataset.classes}'
    )

print('Number of classes:', num_classes)
print('Classes:')
for idx, name in enumerate(class_names):
    print(f'{idx}: {name}')

train_targets = np.array(train_raw_dataset.targets)
eval_targets = np.array(eval_raw_dataset.targets)

print('\nTraining class counts from Augmented Images:')
for idx, name in enumerate(class_names):
    print(f'{name}: {(train_targets == idx).sum()} images')

print('\nValidation/test source class counts from Original Images:')
for idx, name in enumerate(class_names):
    print(f'{name}: {(eval_targets == idx).sum()} images')


Number of classes: 8
Classes:
0: Bacterial Leaf Blight
1: Brown Spot
2: Healthy Rice Leaf
3: Leaf Blast
4: Leaf scald
5: Narrow Brown Leaf Spot
6: Rice Hispa
7: Sheath Blight

Training class counts from Augmented Images:
Bacterial Leaf Blight: 536 images
Brown Spot: 810 images
Healthy Rice Leaf: 511 images
Leaf Blast: 929 images
Leaf scald: 560 images
Narrow Brown Leaf Spot: 353 images
Rice Hispa: 664 images
Sheath Blight: 825 images

Validation/test source class counts from Original Images:
Bacterial Leaf Blight: 180 images
Brown Spot: 267 images
Healthy Rice Leaf: 157 images
Leaf Blast: 305 images
Leaf scald: 189 images
Narrow Brown Leaf Spot: 117 images
Rice Hispa: 215 images
Sheath Blight: 271 images


In [ ]:
def show_sample_images(dataset, class_names, images_per_row=4):
    indices = []
    targets = np.array(dataset.targets)
    for class_idx in range(len(class_names)):
        class_indices = np.where(targets == class_idx)[0]
        if len(class_indices) > 0:
            indices.append(int(class_indices[0]))

    rows = int(np.ceil(len(indices) / images_per_row))
    plt.figure(figsize=(4 * images_per_row, 4 * rows))
    for plot_idx, dataset_idx in enumerate(indices, start=1):
        image, label = dataset[dataset_idx]
        plt.subplot(rows, images_per_row, plot_idx)
        plt.imshow(image)
        plt.title(class_names[label])
        plt.axis('off')
    plt.tight_layout()
    plt.show()


show_sample_images(eval_raw_dataset, class_names)


In [ ]:
IMG_SIZE = 224
RESIZE_SIZE = 280
BATCH_SIZE = 64
NUM_WORKERS = 4
PIN_MEMORY = torch.cuda.is_available()

train_transform = transforms.Compose([
    transforms.Resize(RESIZE_SIZE),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize(RESIZE_SIZE),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


class ImageFolderSubset(Dataset):
    def __init__(self, samples, classes, class_to_idx, transform=None):
        self.samples = list(samples)
        self.classes = classes
        self.class_to_idx = class_to_idx
        self.transform = transform
        self.targets = [label for _, label in self.samples]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        path, label = self.samples[index]
        image = Image.open(path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label


def remap_samples_to_reference(dataset, reference_class_to_idx):
    idx_to_class = {idx: class_name for class_name, idx in dataset.class_to_idx.items()}
    remapped = []
    for path, label in dataset.samples:
        class_name = idx_to_class[label]
        remapped.append((path, reference_class_to_idx[class_name]))
    return remapped


train_samples = train_raw_dataset.samples
eval_samples = remap_samples_to_reference(eval_raw_dataset, train_raw_dataset.class_to_idx)
eval_targets_remapped = np.array([label for _, label in eval_samples])

val_idx, test_idx = train_test_split(
    np.arange(len(eval_samples)),
    test_size=0.5,
    random_state=SEED,
    stratify=eval_targets_remapped,
)

train_dataset = ImageFolderSubset(train_samples, class_names, train_raw_dataset.class_to_idx, train_transform)
val_dataset = ImageFolderSubset([eval_samples[i] for i in val_idx], class_names, train_raw_dataset.class_to_idx, eval_transform)
test_dataset = ImageFolderSubset([eval_samples[i] for i in test_idx], class_names, train_raw_dataset.class_to_idx, eval_transform)

loader_kwargs = {
    'batch_size': BATCH_SIZE,
    'num_workers': NUM_WORKERS,
    'pin_memory': PIN_MEMORY,
}
if NUM_WORKERS > 0:
    loader_kwargs.update({
        'persistent_workers': True,
        'prefetch_factor': 2,
    })

train_loader = DataLoader(train_dataset, shuffle=True, **loader_kwargs)
val_loader = DataLoader(val_dataset, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_kwargs)

print(f'Train images from Augmented Images: {len(train_dataset)}')
print(f'Val images from Original Images:    {len(val_dataset)}')
print(f'Test images from Original Images:   {len(test_dataset)}')


Train images from Augmented Images: 5188
Val images from Original Images:    850
Test images from Original Images:   851


In [ ]:
class ECABlock(nn.Module):
    def __init__(self, channels, gamma=2, b=1):
        super().__init__()
        t = int(abs((math.log2(channels) + b) / gamma))
        kernel_size = t if t % 2 else t + 1

        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv1d(
            in_channels=1,
            out_channels=1,
            kernel_size=kernel_size,
            padding=kernel_size // 2,
            bias=False,
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        y = self.avg_pool(x)
        y = y.squeeze(-1).transpose(-1, -2)
        y = self.conv(y)
        y = y.transpose(-1, -2).unsqueeze(-1)
        y = self.sigmoid(y)
        return x * y.expand_as(x)


def make_classifier(in_features, num_classes, dropout_p=0.1):
    return nn.Sequential(
        nn.Dropout(p=dropout_p),
        nn.Linear(in_features, num_classes),
    )


def get_densenet121_baseline(num_classes, pretrained=True, dropout_p=0.1):
    weights = models.DenseNet121_Weights.IMAGENET1K_V1 if pretrained else None
    model = models.densenet121(weights=weights)
    in_features = model.classifier.in_features
    model.classifier = make_classifier(in_features, num_classes, dropout_p=dropout_p)
    return model


class DenseNet121ECA(nn.Module):
    def __init__(self, num_classes, pretrained=True, dropout_p=0.1):
        super().__init__()
        weights = models.DenseNet121_Weights.IMAGENET1K_V1 if pretrained else None
        base = models.densenet121(weights=weights)
        features = base.features

        self.conv0 = features.conv0
        self.norm0 = features.norm0
        self.relu0 = features.relu0
        self.pool0 = features.pool0

        self.denseblock1 = features.denseblock1
        self.transition1 = features.transition1
        self.denseblock2 = features.denseblock2
        self.transition2 = features.transition2
        self.denseblock3 = features.denseblock3
        self.eca3 = ECABlock(1024)
        self.transition3 = features.transition3
        self.denseblock4 = features.denseblock4
        self.eca4 = ECABlock(1024)
        self.norm5 = features.norm5

        self.classifier = make_classifier(
            base.classifier.in_features,
            num_classes,
            dropout_p=dropout_p,
        )

    def forward(self, x):
        x = self.conv0(x)
        x = self.norm0(x)
        x = self.relu0(x)
        x = self.pool0(x)

        x = self.denseblock1(x)
        x = self.transition1(x)
        x = self.denseblock2(x)
        x = self.transition2(x)
        x = self.eca3(self.denseblock3(x))
        x = self.transition3(x)
        x = self.eca4(self.denseblock4(x))
        x = self.norm5(x)
        x = torch.relu(x)
        x = torch.nn.functional.adaptive_avg_pool2d(x, (1, 1))
        x = torch.flatten(x, 1)
        return self.classifier(x)


def get_densenet121_eca(num_classes, pretrained=True, dropout_p=0.1):
    return DenseNet121ECA(num_classes, pretrained=pretrained, dropout_p=dropout_p)


def get_resnet50(num_classes, pretrained=True, dropout_p=0.1):
    weights = models.ResNet50_Weights.DEFAULT if pretrained else None
    model = models.resnet50(weights=weights)
    in_features = model.fc.in_features
    model.fc = make_classifier(in_features, num_classes, dropout_p=dropout_p)
    return model


def get_mobilenet_v3_large(num_classes, pretrained=True, dropout_p=0.1):
    weights = models.MobileNet_V3_Large_Weights.DEFAULT if pretrained else None
    model = models.mobilenet_v3_large(weights=weights)
    hidden_features = model.classifier[-1].in_features
    model.classifier[-2] = nn.Dropout(p=dropout_p, inplace=True)
    model.classifier[-1] = nn.Linear(hidden_features, num_classes)
    return model


def get_efficientnet_b0(num_classes, pretrained=True, dropout_p=0.1):
    weights = models.EfficientNet_B0_Weights.DEFAULT if pretrained else None
    model = models.efficientnet_b0(weights=weights)
    in_features = model.classifier[-1].in_features
    model.classifier = make_classifier(in_features, num_classes, dropout_p=dropout_p)
    return model


try:
    import timm
except ImportError:
    import subprocess
    import sys

    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'timm'])
    import timm


def get_pretrained_ghostnet(num_classes, pretrained=True, dropout_p=0.1):
    return timm.create_model(
        'ghostnet_100',
        pretrained=pretrained,
        num_classes=num_classes,
        drop_rate=dropout_p,
    )


def find_last_conv2d(model):
    last_conv = None
    for module in model.modules():
        if isinstance(module, nn.Conv2d):
            last_conv = module
    if last_conv is None:
        raise ValueError(f'No Conv2d layer found in {model.__class__.__name__}')
    return last_conv


MODEL_CONFIGS = [
    {
        'key': 'densenet121_baseline',
        'display_name': 'DenseNet121 Baseline',
        'factory': get_densenet121_baseline,
        'target_layer': lambda model: model.features.denseblock4,
    },
    {
        'key': 'densenet121_eca',
        'display_name': 'DenseNet121-ECA',
        'factory': get_densenet121_eca,
        'target_layer': lambda model: model.denseblock4,
    },
    {
        'key': 'resnet50',
        'display_name': 'ResNet50',
        'factory': get_resnet50,
        'target_layer': lambda model: model.layer4[-1],
    },
    {
        'key': 'mobilenet_v3_large',
        'display_name': 'MobileNetV3-Large',
        'factory': get_mobilenet_v3_large,
        'target_layer': lambda model: model.features[-1],
    },
    {
        'key': 'efficientnet_b0',
        'display_name': 'EfficientNet-B0',
        'factory': get_efficientnet_b0,
        'target_layer': lambda model: model.features[-1],
    },
    {
        'key': 'ghostnet_100_timm_pretrained',
        'display_name': 'GhostNet-timm-pretrained',
        'factory': get_pretrained_ghostnet,
        'target_layer': lambda model: find_last_conv2d(model),
    },
]


for config in MODEL_CONFIGS:
    preview_model = config['factory'](num_classes, dropout_p=0.1)
    print(f"{config['display_name']} parameters: {sum(p.numel() for p in preview_model.parameters()):,}")
    del preview_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 195MB/s]


DenseNet121 Baseline parameters: 6,962,056
DenseNet121-ECA parameters: 6,962,066
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 243MB/s]


ResNet50 parameters: 23,524,424
Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-5c1a4163.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_large-5c1a4163.pth


...

100%|██████████| 20.5M/20.5M [00:00<00:00, 84.0MB/s]


EfficientNet-B0 parameters: 4,017,796


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/20.9M [00:00<?, ?B/s]

GhostNet-timm-pretrained parameters: 3,911,756


In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None, reduction='mean'):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.reduction = reduction

    def forward(self, logits, targets):
        ce_loss = torch.nn.functional.cross_entropy(
            logits,
            targets,
            weight=self.alpha,
            reduction='none',
        )
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        if self.reduction == 'sum':
            return focal_loss.sum()
        return focal_loss


def make_class_weight_tensor(dataset, num_classes, device):
    targets = np.array(dataset.targets)
    counts = np.bincount(targets, minlength=num_classes)
    weights = counts.sum() / np.maximum(counts, 1)
    weights = weights / weights.mean()
    print('Focal loss class weights:')
    for idx, name in enumerate(class_names):
        print(f'{name}: count={counts[idx]}, weight={weights[idx]:.4f}')
    return torch.tensor(weights, dtype=torch.float32, device=device)


def autocast_context(use_amp):
    if use_amp and torch.cuda.is_available():
        return torch.amp.autocast('cuda')
    return nullcontext()


def make_grad_scaler(use_amp):
    try:
        return torch.amp.GradScaler('cuda', enabled=use_amp)
    except TypeError:
        return torch.cuda.amp.GradScaler(enabled=use_amp)


def train_one_epoch(model, loader, criterion, optimizer, device, scaler=None, use_amp=False, epoch=None, total_epochs=None, model_name='Model'):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    desc = f'{model_name} Train'
    if epoch is not None and total_epochs is not None:
        desc = f'{model_name} Train {epoch}/{total_epochs}'

    progress = tqdm(loader, desc=desc, leave=False)
    for images, labels in progress:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast_context(use_amp):
            outputs = model(images)
            loss = criterion(outputs, labels)

        if scaler is not None and use_amp:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        batch_size = images.size(0)
        running_loss += loss.item() * batch_size
        correct += (outputs.argmax(dim=1) == labels).sum().item()
        total += batch_size
        progress.set_postfix(loss=running_loss / total, acc=correct / total)

    return running_loss / total, correct / total


@torch.no_grad()
def validate_one_epoch(model, loader, criterion, device, use_amp=False, epoch=None, total_epochs=None, model_name='Model'):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    desc = f'{model_name} Val'
    if epoch is not None and total_epochs is not None:
        desc = f'{model_name} Val {epoch}/{total_epochs}'

    progress = tqdm(loader, desc=desc, leave=False)
    for images, labels in progress:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with autocast_context(use_amp):
            outputs = model(images)
            loss = criterion(outputs, labels)

        batch_size = images.size(0)
        running_loss += loss.item() * batch_size
        correct += (outputs.argmax(dim=1) == labels).sum().item()
        total += batch_size
        progress.set_postfix(loss=running_loss / total, acc=correct / total)

    return running_loss / total, correct / total


def train_model(
    model,
    model_name,
    train_loader,
    val_loader,
    epochs=20,
    lr=5e-5,
    weight_decay=5e-4,
    criterion=None,
    early_stopping_patience=5,
    early_stopping_min_delta=0.0,
):
    if criterion is None:
        criterion = FocalLoss()

    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

    use_amp = torch.cuda.is_available()
    scaler = make_grad_scaler(use_amp)

    print(f'\nTraining {model_name}')
    print(f'AMP mixed precision enabled: {use_amp}')
    print(f'L2 regularization / weight decay: {weight_decay}')
    print(f'Early stopping patience: {early_stopping_patience}')

    history = {
        'train_loss': [],
        'train_acc': [],
        'val_loss': [],
        'val_acc': [],
        'lr': [],
    }

    best_val_acc = -np.inf
    best_epoch = 0
    no_improve = 0
    checkpoint_path = CHECKPOINT_DIR / f'{model_name}.pt'
    start_time = time.time()

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            device,
            scaler=scaler,
            use_amp=use_amp,
            epoch=epoch,
            total_epochs=epochs,
            model_name=model_name,
        )
        val_loss, val_acc = validate_one_epoch(
            model,
            val_loader,
            criterion,
            device,
            use_amp=use_amp,
            epoch=epoch,
            total_epochs=epochs,
            model_name=model_name,
        )

        scheduler.step(val_loss)
        current_lr = optimizer.param_groups[0]['lr']

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['lr'].append(current_lr)

        improved = val_acc > best_val_acc + early_stopping_min_delta
        if improved:
            best_val_acc = val_acc
            best_epoch = epoch
            no_improve = 0
            torch.save(
                {
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'best_val_acc': best_val_acc,
                    'best_epoch': best_epoch,
                    'history': history,
                    'model_name': model_name,
                },
                checkpoint_path,
            )
        else:
            no_improve += 1

        print(
            f'[{model_name}] Epoch {epoch:02d}/{epochs} | '
            f'train loss {train_loss:.4f} acc {train_acc:.4f} | '
            f'val loss {val_loss:.4f} acc {val_acc:.4f} | '
            f'best val acc {best_val_acc:.4f} | no improve {no_improve}'
        )

        if early_stopping_patience is not None and no_improve >= early_stopping_patience:
            print(f'Early stopping triggered at epoch {epoch}. Best validation accuracy: {best_val_acc:.4f}')
            break

    elapsed_min = (time.time() - start_time) / 60
    print(f'{model_name} training finished in {elapsed_min:.2f} minutes. Best val acc: {best_val_acc:.4f}')
    return history, checkpoint_path, best_val_acc, best_epoch


In [ ]:
EPOCHS = 20
LEARNING_RATE = 5e-5
WEIGHT_DECAY = 5e-4
DROPOUT_P = 0.1
EARLY_STOPPING_PATIENCE = 5
EARLY_STOPPING_MIN_DELTA = 0.0
FOCAL_GAMMA = 2.0
USE_FOCAL_CLASS_WEIGHTS = True

print(f'Dropout p: {DROPOUT_P}')
print(f'L2 regularization / weight decay: {WEIGHT_DECAY}')
print(f'Early stopping patience: {EARLY_STOPPING_PATIENCE}')
print(f'Focal Loss gamma: {FOCAL_GAMMA}')

focal_alpha = make_class_weight_tensor(train_dataset, num_classes, device) if USE_FOCAL_CLASS_WEIGHTS else None
focal_criterion = FocalLoss(gamma=FOCAL_GAMMA, alpha=focal_alpha)


Dropout p: 0.1
L2 regularization / weight decay: 0.0005
Early stopping patience: 5
Focal Loss gamma: 2.0
Focal loss class weights:
Bacterial Leaf Blight: count=536, weight=1.1080
Brown Spot: count=810, weight=0.7332
Healthy Rice Leaf: count=511, weight=1.1622
Leaf Blast: count=929, weight=0.6393
Leaf scald: count=560, weight=1.0605
Narrow Brown Leaf Spot: count=353, weight=1.6824
Rice Hispa: count=664, weight=0.8944
Sheath Blight: count=825, weight=0.7199


In [ ]:
experiment_runs = []

for config in MODEL_CONFIGS:
    set_seed(SEED)
    model = config['factory'](num_classes, dropout_p=DROPOUT_P).to(device)
    model_name = f"{config['key']}_focal_dropout_l2"

    history, checkpoint, best_val_acc, best_val_epoch = train_model(
        model,
        model_name,
        train_loader,
        val_loader,
        epochs=EPOCHS,
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        criterion=focal_criterion,
        early_stopping_patience=EARLY_STOPPING_PATIENCE,
        early_stopping_min_delta=EARLY_STOPPING_MIN_DELTA,
    )

    experiment_runs.append({
        'key': config['key'],
        'display_name': config['display_name'],
        'factory': config['factory'],
        'target_layer': config['target_layer'],
        'history': history,
        'checkpoint': checkpoint,
        'best_val_acc': best_val_acc,
        'best_val_epoch': best_val_epoch,
        'final_train_loss': history['train_loss'][-1],
        'final_val_loss': history['val_loss'][-1],
    })

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f'Finished training {len(experiment_runs)} models.')



Training densenet121_baseline_focal_dropout_l2
AMP mixed precision enabled: True
L2 regularization / weight decay: 0.0005
Early stopping patience: 5


densenet121_baseline_focal_dropout_l2 Train 1/20:   0%|          | 0/82 [00:00<?, ?it/s]

densenet121_baseline_focal_dropout_l2 Val 1/20:   0%|          | 0/14 [00:00<?, ?it/s]

[densenet121_baseline_focal_dropout_l2] Epoch 01/20 | train loss 0.6404 acc 0.6754 | val loss 0.3046 acc 0.7988 | best val acc 0.7988 | no improve 0


densenet121_baseline_focal_dropout_l2 Train 2/20:   0%|          | 0/82 [00:00<?, ?it/s]

...

ghostnet_100_timm_pretrained_focal_dropout_l2 Val 15/20:   0%|          | 0/14 [00:00<?, ?it/s]

[ghostnet_100_timm_pretrained_focal_dropout_l2] Epoch 15/20 | train loss 0.0037 acc 0.9971 | val loss 0.1638 acc 0.9165 | best val acc 0.9224 | no improve 4


ghostnet_100_timm_pretrained_focal_dropout_l2 Train 16/20:   0%|          | 0/82 [00:00<?, ?it/s]

ghostnet_100_timm_pretrained_focal_dropout_l2 Val 16/20:   0%|          | 0/14 [00:00<?, ?it/s]

[ghostnet_100_timm_pretrained_focal_dropout_l2] Epoch 16/20 | train loss 0.0035 acc 0.9979 | val loss 0.1696 acc 0.9188 | best val acc 0.9224 | no improve 5
Early stopping triggered at epoch 16. Best validation accuracy: 0.9224
ghostnet_100_timm_pretrained_focal_dropout_l2 training finished in 17.67 minutes. Best val acc: 0.9224
Finished training 6 models.


In [ ]:
def safe_filename(name):
    return name.lower().replace(' ', '_').replace('-', '_').replace('+', 'plus')


def plot_history(history, model_name):
    epochs = range(1, len(history['train_loss']) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(epochs, history['train_loss'], label='Train Loss')
    axes[0].plot(epochs, history['val_loss'], label='Val Loss')
    axes[0].set_title(f'{model_name} Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(epochs, history['train_acc'], label='Train Accuracy')
    axes[1].plot(epochs, history['val_acc'], label='Val Accuracy')
    axes[1].set_title(f'{model_name} Accuracy')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    save_path = PLOT_DIR / f'{safe_filename(model_name)}_history.png'
    plt.savefig(save_path, dpi=200, bbox_inches='tight')
    plt.show()
    print('Saved:', save_path)


for run in experiment_runs:
    plot_history(run['history'], run['display_name'])


In [ ]:
@torch.no_grad()
def predict(model, loader, device, use_amp=None):
    model.eval()
    all_preds = []
    all_labels = []
    if use_amp is None:
        use_amp = torch.cuda.is_available()

    for images, labels in tqdm(loader, desc='Predict', leave=False):
        images = images.to(device, non_blocking=True)
        with autocast_context(use_amp):
            outputs = model(images)
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.numpy().tolist())

    return np.array(all_labels), np.array(all_preds)


def load_checkpoint(model, checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    return model


def evaluate_model(model, model_name, checkpoint_path, metadata=None):
    model = load_checkpoint(model, checkpoint_path)
    y_true, y_pred = predict(model, test_loader, device)

    accuracy = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average='weighted',
        zero_division=0,
    )

    report_text = classification_report(y_true, y_pred, target_names=class_names, zero_division=0)
    report_dict = classification_report(y_true, y_pred, target_names=class_names, zero_division=0, output_dict=True)

    print(f'\n{model_name} classification report')
    print(report_text)

    safe_name = safe_filename(model_name)
    report_path = REPORT_DIR / f'{safe_name}_classification_report.txt'
    report_path.write_text(report_text, encoding='utf-8')

    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(10, 10))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(ax=ax, xticks_rotation=45, cmap='Blues', colorbar=False)
    ax.set_title(f'{model_name} Confusion Matrix')
    plt.tight_layout()
    cm_path = PLOT_DIR / f'{safe_name}_confusion_matrix.png'
    plt.savefig(cm_path, dpi=200, bbox_inches='tight')
    plt.show()
    print('Saved:', cm_path)
    print('Saved:', report_path)

    result = {
        'model': model_name,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'dropout_p': DROPOUT_P,
        'weight_decay': WEIGHT_DECAY,
        'checkpoint': str(checkpoint_path),
    }
    if metadata:
        result.update(metadata)

    per_class_rows = []
    for class_name in class_names:
        row = {
            'model': model_name,
            'class': class_name,
            'precision': report_dict[class_name]['precision'],
            'recall': report_dict[class_name]['recall'],
            'f1_score': report_dict[class_name]['f1-score'],
            'support': report_dict[class_name]['support'],
        }
        per_class_rows.append(row)
        result[f"f1_{safe_filename(class_name)}"] = report_dict[class_name]['f1-score']

    return result, per_class_rows


evaluation_results = []
per_class_results = []

for run in experiment_runs:
    set_seed(SEED)
    model = run['factory'](num_classes, dropout_p=DROPOUT_P).to(device)
    metadata = {
        'best_val_acc': run['best_val_acc'],
        'best_val_epoch': run['best_val_epoch'],
        'final_train_loss': run['final_train_loss'],
        'final_val_loss': run['final_val_loss'],
    }
    result, per_class_rows = evaluate_model(model, run['display_name'], run['checkpoint'], metadata=metadata)
    evaluation_results.append(result)
    per_class_results.extend(per_class_rows)
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


In [ ]:
try:
    import pandas as pd

    comparison_df = pd.DataFrame(evaluation_results)
    comparison_df = comparison_df.sort_values('f1_score', ascending=False).reset_index(drop=True)
    display(comparison_df)
    comparison_df.to_csv(OUTPUT_DIR / 'model_comparison_all_six_focal_dropout_l2.csv', index=False)

    per_class_df = pd.DataFrame(per_class_results)
    display(per_class_df.sort_values(['class', 'f1_score'], ascending=[True, False]).reset_index(drop=True))
    per_class_df.to_csv(OUTPUT_DIR / 'per_class_metrics_all_six_focal_dropout_l2.csv', index=False)
except ImportError:
    comparison_df = None
    per_class_df = None
    for row in sorted(evaluation_results, key=lambda x: x['f1_score'], reverse=True):
        print(row)

best_model = max(evaluation_results, key=lambda x: x['f1_score'])
print(f"Best model by weighted F1-score: {best_model['model']} ({best_model['f1_score']:.4f})")
print(f'Dropout p: {DROPOUT_P}')
print(f'L2 regularization / weight decay: {WEIGHT_DECAY}')
print(f'Focal Loss gamma: {FOCAL_GAMMA}')


,model,accuracy,precision,recall,f1_score,dropout_p,weight_decay,checkpoint,best_val_acc,best_val_epoch,final_train_loss,final_val_loss,f1_bacterial_leaf_blight,f1_brown_spot,f1_healthy_rice_leaf,f1_leaf_blast,f1_leaf_scald,f1_narrow_brown_leaf_spot,f1_rice_hispa,f1_sheath_blight
0,DenseNet121-ECA,0.960047,0.960512,0.960047,0.960140,0.1,0.0005,/content/outputs_all_models_focal/checkpoints/...,0.948235,9,0.001286,0.130507,0.928962,0.973783,0.981132,0.957377,0.934783,0.881356,0.990654,0.985294
1,MobileNetV3-Large,0.954172,0.954623,0.954172,0.954243,0.1,0.0005,/content/outputs_all_models_focal/checkpoints/...,0.941176,11,0.004343,0.129923,0.921348,0.961832,0.987342,0.954545,0.936170,0.873950,0.986175,0.970588
2,EfficientNet-B0,0.951821,0.952436,0.951821,0.951910,0.1,0.0005,/content/outputs_all_models_focal/checkpoints/...,0.944706,15,0.005786,0.105096,0.913978,0.954198,0.980645,0.950495,0.925532,0.896552,0.990826,0.970803
3,DenseNet121 Baseline,0.951821,0.952466,0.951821,0.951905,0.1,0.0005,/content/outputs_all_models_focal/checkpoints/...,0.943529,6,0.002421,0.127225,0.934783,0.961832,0.962500,0.938511,0.931217,0.894737,0.981308,0.977778
4,ResNet50,0.950646,0.950929,0.950646,0.950668,0.1,0.0005,/content/outputs_all_models_focal/checkpoints/...,0.942353,7,0.002074,0.130861,0.913043,0.962121,0.987342,0.951140,0.903226,0.862069,0.995392,0.977778
5,GhostNet-timm-pretrained,0.947121,0.947457,0.947121,0.947173,0.1,0.0005,/content/outputs_all_models_focal/checkpoints/...,0.922353,11,0.003506,0.169581,0.916201,0.954198,0.968153,0.951140,0.910053,0.898305,0.981481,0.963504


,model,class,precision,recall,f1_score,support
0,DenseNet121 Baseline,Bacterial Leaf Blight,0.914894,0.955556,0.934783,90.0
1,DenseNet121-ECA,Bacterial Leaf Blight,0.913978,0.944444,0.928962,90.0
2,MobileNetV3-Large,Bacterial Leaf Blight,0.931818,0.911111,0.921348,90.0
3,GhostNet-timm-pretrained,Bacterial Leaf Blight,0.921348,0.911111,0.916201,90.0
4,EfficientNet-B0,Bacterial Leaf Blight,0.885417,0.944444,0.913978,90.0
5,ResNet50,Bacterial Leaf Blight,0.893617,0.933333,0.913043,90.0
6,DenseNet121-ECA,Brown Spot,0.977444,0.970149,0.973783,134.0
7,ResNet50,Brown Spot,0.976923,0.947761,0.962121,134.0
8,DenseNet121 Baseline,Brown Spot,0.984375,0.940299,0.961832,134.0
9,MobileNetV3-Large,Brown Spot,0.984375,0.940299,0.961832,134.0


Best model by weighted F1-score: DenseNet121-ECA (0.9601)
Dropout p: 0.1
L2 regularization / weight decay: 0.0005
Focal Loss gamma: 2.0


In [ ]:
class GradCAM:
    """Grad-CAM implementation that avoids register_full_backward_hook.

    PyTorch can raise "BackwardHookFunctionBackward is a view and is being
    modified inplace" when full backward hooks are attached to layers followed
    by inplace activations. This version registers a hook directly on the target
    tensor instead, and stores cloned activations/gradients.
    """

    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.activations = None
        self.gradients = None
        self.forward_handle = target_layer.register_forward_hook(self._forward_hook)

    def _forward_hook(self, module, inputs, output):
        self.activations = output.detach().clone()

        def _save_gradient(grad):
            self.gradients = grad.detach().clone()

        if output.requires_grad:
            output.register_hook(_save_gradient)

    def generate(self, image_tensor, class_idx=None):
        self.model.eval()
        self.model.zero_grad(set_to_none=True)
        self.activations = None
        self.gradients = None

        output = self.model(image_tensor)
        probabilities = torch.softmax(output, dim=1)
        if class_idx is None:
            class_idx = int(output.argmax(dim=1).item())

        score = output[:, class_idx].sum()
        score.backward()

        if self.activations is None or self.gradients is None:
            raise RuntimeError(
                'Grad-CAM hooks did not capture activations/gradients. '
                'Try selecting an earlier convolutional target layer.'
            )

        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = torch.relu(cam)
        cam = torch.nn.functional.interpolate(
            cam,
            size=image_tensor.shape[-2:],
            mode='bilinear',
            align_corners=False,
        )
        cam = cam.squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        confidence = float(probabilities[0, class_idx].detach().cpu().item())
        return cam, class_idx, confidence

    def close(self):
        self.forward_handle.remove()


def disable_inplace_activations(module):
    for child in module.modules():
        if hasattr(child, 'inplace'):
            child.inplace = False


def unnormalize(image_tensor):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    image = image_tensor.cpu() * std + mean
    return image.clamp(0, 1).permute(1, 2, 0).numpy()


def first_index_per_class(dataset, num_classes):
    selected = {}
    for idx, label in enumerate(dataset.targets):
        if label not in selected:
            selected[label] = idx
        if len(selected) == num_classes:
            break
    return [selected[class_idx] for class_idx in range(num_classes) if class_idx in selected]


def show_gradcam_examples(model, target_layer, model_name, dataset, num_images=None):
    model.eval()
    disable_inplace_activations(model)
    gradcam = GradCAM(model, target_layer)
    indices = first_index_per_class(dataset, num_classes)
    if num_images is not None:
        indices = indices[:num_images]

    fig, axes = plt.subplots(len(indices), 3, figsize=(12, 4 * len(indices)))
    if len(indices) == 1:
        axes = np.expand_dims(axes, axis=0)

    for row, idx in enumerate(indices):
        image_tensor, true_label = dataset[idx]
        input_tensor = image_tensor.unsqueeze(0).to(device)
        cam, pred_label, confidence = gradcam.generate(input_tensor)
        original = unnormalize(image_tensor)

        axes[row, 0].imshow(original)
        axes[row, 0].set_title(f'True: {class_names[true_label]}')
        axes[row, 0].axis('off')

        axes[row, 1].imshow(cam, cmap='jet')
        axes[row, 1].set_title('Grad-CAM')
        axes[row, 1].axis('off')

        axes[row, 2].imshow(original)
        axes[row, 2].imshow(cam, cmap='jet', alpha=0.45)
        axes[row, 2].set_title(f'Pred: {class_names[pred_label]} ({confidence:.2f})')
        axes[row, 2].axis('off')

    fig.suptitle(f'{model_name} Grad-CAM Examples', fontsize=16)
    plt.tight_layout()
    save_path = PLOT_DIR / f'{safe_filename(model_name)}_gradcam.png'
    plt.savefig(save_path, dpi=200, bbox_inches='tight')
    plt.show()
    print('Saved:', save_path)
    gradcam.close()


RUN_GRADCAM = True

if RUN_GRADCAM:
    for run in experiment_runs:
        set_seed(SEED)
        model = run['factory'](num_classes, dropout_p=DROPOUT_P).to(device)
        model = load_checkpoint(model, run['checkpoint'])
        disable_inplace_activations(model)
        target_layer = run['target_layer'](model)
        show_gradcam_examples(model, target_layer, run['display_name'], test_dataset)
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


In [ ]:
comparison = evaluation_results

try:
    import pandas as pd
    comparison_df = pd.DataFrame(comparison)
    comparison_df = comparison_df.sort_values('f1_score', ascending=False).reset_index(drop=True)
    display(comparison_df)
    comparison_df.to_csv(OUTPUT_DIR / 'model_comparison_all_five_focal_dropout_l2.csv', index=False)
except ImportError:
    comparison_df = None
    for row in sorted(comparison, key=lambda x: x['f1_score'], reverse=True):
        print(row)

best_model = max(comparison, key=lambda x: x['f1_score'])
print(f"Best model by weighted F1-score: {best_model['model']} ({best_model['f1_score']:.4f})")
print(f"Dropout p: {DROPOUT_P}")
print(f"L2 regularization / weight decay: {WEIGHT_DECAY}")


,model,accuracy,precision,recall,f1_score,dropout_p,weight_decay,checkpoint,best_val_acc,best_val_epoch,final_train_loss,final_val_loss,f1_bacterial_leaf_blight,f1_brown_spot,f1_healthy_rice_leaf,f1_leaf_blast,f1_leaf_scald,f1_narrow_brown_leaf_spot,f1_rice_hispa,f1_sheath_blight
0,DenseNet121-ECA,0.960047,0.960512,0.960047,0.960140,0.1,0.0005,/content/outputs_all_models_focal/checkpoints/...,0.948235,9,0.001286,0.130507,0.928962,0.973783,0.981132,0.957377,0.934783,0.881356,0.990654,0.985294
1,MobileNetV3-Large,0.954172,0.954623,0.954172,0.954243,0.1,0.0005,/content/outputs_all_models_focal/checkpoints/...,0.941176,11,0.004343,0.129923,0.921348,0.961832,0.987342,0.954545,0.936170,0.873950,0.986175,0.970588
2,EfficientNet-B0,0.951821,0.952436,0.951821,0.951910,0.1,0.0005,/content/outputs_all_models_focal/checkpoints/...,0.944706,15,0.005786,0.105096,0.913978,0.954198,0.980645,0.950495,0.925532,0.896552,0.990826,0.970803
3,DenseNet121 Baseline,0.951821,0.952466,0.951821,0.951905,0.1,0.0005,/content/outputs_all_models_focal/checkpoints/...,0.943529,6,0.002421,0.127225,0.934783,0.961832,0.962500,0.938511,0.931217,0.894737,0.981308,0.977778
4,ResNet50,0.950646,0.950929,0.950646,0.950668,0.1,0.0005,/content/outputs_all_models_focal/checkpoints/...,0.942353,7,0.002074,0.130861,0.913043,0.962121,0.987342,0.951140,0.903226,0.862069,0.995392,0.977778
5,GhostNet-timm-pretrained,0.947121,0.947457,0.947121,0.947173,0.1,0.0005,/content/outputs_all_models_focal/checkpoints/...,0.922353,11,0.003506,0.169581,0.916201,0.954198,0.968153,0.951140,0.910053,0.898305,0.981481,0.963504


Best model by weighted F1-score: DenseNet121-ECA (0.9601)
Dropout p: 0.1
L2 regularization / weight decay: 0.0005
